In [27]:
import os
from datetime import datetime, timedelta

import colorcet as cc
import contextily as cx
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import HoverTool, Title
from pathlib import Path
import folium
import rioxarray
import random
import re

from blackmarble.extract import bm_extract
from blackmarble.raster import bm_raster
from typing import List


import xarray as xr
import numpy as np

%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')
from ntl_functions import plot_NASA_NTL, filter_dataset_by_bounding_box, mask_dataset_by_geometry

plt.rcParams["figure.figsize"] = (18, 10)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
os.getcwd()
#os.chdir("C:/Users/samsa/OneDrive - ODI/Central Bank Somaliland/Night Lights")
#base_directory = Path("C:/Users/samsa/OneDrive/Documents/Central Bank Somaliland/Night Lights")
base_directory = Path("C:\\Users\\samsa\\Documents")


# Define the specific output directory within the base directory
output_directory = base_directory / "black_marble_output_full"

# Ensure the output directory exists
output_directory.mkdir(parents=True, exist_ok=True)

print(f"Output Directory: {output_directory.resolve()}")

Output Directory: C:\Users\samsa\Documents\black_marble_output_full


In [ ]:
somaliland_shp_berbera = gpd.read_file(
    "../data/Combined_Datasets/Somaliland_with_berbera/somaliland_with_berbera.shp"
)

C:\Users\samsa\AppData\Local\Temp\ipykernel_2836\4156978314.py:18: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  expanded_borders = somaliland_shp.dissolve().buffer(0.0027)


In [ ]:
def parse_vdn_files(file_string, utc_offset=0):
    # Clean up the raw string and split into filenames
    files = file_string.strip("b'").strip("'").split(",")
    
    results = []
    for f in files:
        filename = f.split("/")[-1]  # just the filename, not the full path
        
        # Only keep NPP_VDNES_L1 files
        if "NPP_VDNES_L1" in filename:
            try:
                # Example: NPP_VDNES_L1.A2012194.2212.001.2016312152556.hdf
                parts = filename.split(".")
                date_code = parts[1]   # A2012194
                time_code = parts[2]   # 2212
                
                # Convert date and time
                year = int(date_code[1:5])       # 2012
                day_of_year = int(date_code[5:]) # 194
                hour = int(time_code[:2])        # 22
                minute = int(time_code[2:])      # 12
                
                obs_time = datetime.strptime(f"{year}{day_of_year:03d} {hour:02d}{minute:02d}", "%Y%j %H%M")
                local_time = obs_time + timedelta(hours=utc_offset)

                
                results.append({
                    "filename": filename,
                    "observation_time_utc": obs_time.strftime("%Y-%m-%d %H:%M"),
                    "local_time": local_time.strftime("%Y-%m-%d %H:%M")
                })
            except Exception as e:
                print(f"Could not parse {filename}: {e}")
    
    return results

In [52]:
start_year = 2020
end_year = 2022
samples_per_year = 4

# Dictionary to store results
random_days_4_per_year = []

# Loop through each year
for year in range(start_year, end_year + 1):
    start_date = datetime(year, 1, 1)
    end_date = datetime(year, 12, 31)
    delta_days = (end_date - start_date).days

    # Generate unique random days
    random_days = random.sample(range(delta_days + 1), samples_per_year)
    random_dates = [
        (start_date + timedelta(days=day)).strftime("%Y-%m-%d")
        for day in sorted(random_days)]

    random_days_4_per_year.extend(random_dates)

In [ ]:
date_times = []
for day in random_days_4_per_year:
    data_single_day = bm_raster(
        somaliland_shp_berbera, 
        product_id="VNP46A1", 
        date_range=day, 
        bearer=bearer)
    date_times.append(parse_vdn_files(data_single_day.attrs["InputPointer"], utc_offset=3))
    

OBTAINING MANIFEST...:   0%|          | 0/2 [00:00<?, ?it/s]

QUEUEING TASKS | Downloading (86.8 MB)...:   0%|          | 0/2 [00:00<?, ?file/s]

PROCESSING TASKS | Downloading (86.8 MB)...:   0%|          | 0/2 [00:00<?, ?file/s]

COLLECTING RESULTS | Downloading (86.8 MB)...:   0%|          | 0/2 [00:00<?, ?file/s]

TypeError: argument of type 'NoneType' is not iterable

In [55]:
date_times

[]

In [21]:
df = pd.DataFrame(date_times)
df.to_csv("overpass_times_2019.csv", index=False)

In [21]:
def extract_viirs_times(raw_string, utc_offset=0):
    """
    Extract overpass times from a raw MODAPS-style string.

    Parameters
    ----------
    raw_string : str
        String containing multiple comma-separated VIIRS file paths.
    utc_offset : int
        Hours to shift from UTC (e.g. +3 for Somaliland).

    Returns
    -------
    list of dict
        Each dict contains: filename, utc_time, local_time.
    """

    # Remove the leading b' and trailing quote if present
    clean_str = raw_string.strip("b'").strip("'")

    # Split into filenames
    filenames = clean_str.split(",")

    results = []
    for f in filenames:
        match = re.search(r"A(\d{7})\.(\d{4})", f)
        if match:
            year_doy = match.group(1)   # e.g. 2012194
            hhmm = match.group(2)       # e.g. 2212

            year = int(year_doy[:4])
            doy = int(year_doy[4:])
            hour = int(hhmm[:2])
            minute = int(hhmm[2:])

            # Convert DOY to calendar date
            date = datetime(year, 1, 1) + timedelta(days=doy - 1)
            utc_time = datetime(year, date.month, date.day, hour, minute)

            # Local time adjustment
            local_time = utc_time + timedelta(hours=utc_offset)

            results.append({
                "filename": f.split("/")[-1],
                "local_time": local_time.strftime("%Y-%m-%d %H:%M")
            })

    return results